In [6]:
import pandas as pd

# Proje klasörümüzdeki veriyi yüklüyoruz
df = pd.read_csv('aml_bank_transactions.csv')

# AML Kuralı: 5.000 üzeri tutarlar
suspicious = df[(df['Transaction_Amount'] > 5000) & (df['Status'] == 'Success')]

print("==========================================")
print(f" VERİ SETİ BAŞARIYLA YÜKLENDİ!")
print(f"Toplam İşlem Sayısı: {len(df)}")
print(f"TESPİT EDİLEN YÜKSEK RİSKLİ İŞLEM: {len(suspicious)}")
print("==========================================\n")

suspicious.head()

✅ VERİ SETİ BAŞARIYLA YÜKLENDİ!
Toplam İşlem Sayısı: 500
🚨 TESPİT EDİLEN YÜKSEK RİSKLİ İŞLEM: 41



,Transaction_ID,Sender_Account_ID,Receiver_Account_ID,Transaction_Amount,Transaction_Type,Status,Timestamp
0,TXN100000,ACC1038,ACC1047,5476.88,Transfer,Success,2026-01-01 00:00:00
13,TXN100013,ACC1039,ACC1009,11846.34,Withdrawal,Success,2026-01-01 03:15:00
16,TXN100016,ACC1021,ACC1000,5037.38,Transfer,Success,2026-01-01 04:00:00
27,TXN100027,ACC1043,ACC1026,6933.67,Deposit,Success,2026-01-01 06:45:00
36,TXN100036,ACC1043,ACC1036,5426.92,Deposit,Success,2026-01-01 09:00:00


In [7]:
# 2. AML Senaryosu: En çok para transferi yapan ilk 5 hesap (Mule Account Analysis)
mule_accounts = df[df['Status'] == 'Success'].groupby('Sender_Account_ID').agg(
    Toplam_Islem_Adedi=('Transaction_ID', 'count'),
    Toplam_Transfer_Tutari=('Transaction_Amount', 'sum'),
    Ortalama_Tutar=('Transaction_Amount', 'mean')
).reset_index().sort_values(by='Toplam_Transfer_Tutari', ascending=False)

print("🚨 ŞÜPHELİ PARASAL HACME SAHİP İLK 5 GÖNDEREN HESAP:")
mule_accounts.head(5)

🚨 ŞÜPHELİ PARASAL HACME SAHİP İLK 5 GÖNDEREN HESAP:


,Sender_Account_ID,Toplam_Islem_Adedi,Toplam_Transfer_Tutari,Ortalama_Tutar
31,ACC1031,15,55303.73,3686.915333
32,ACC1032,16,54088.11,3380.506875
24,ACC1024,12,36186.45,3015.537500
34,ACC1034,12,35044.93,2920.410833
39,ACC1039,9,33697.73,3744.192222


In [8]:
# 3. AML Senaryosu: En çok para toplayan alıcı hesaplar (High-Volume Receivers)
suspicious_receivers = df[df['Status'] == 'Success'].groupby('Receiver_Account_ID').agg(
    Gelen_Islem_Adedi=('Transaction_ID', 'count'),
    Toplam_Gelen_Tutar=('Transaction_Amount', 'sum')
).reset_index().sort_values(by='Toplam_Gelen_Tutar', ascending=False)

print("🚨 EN ÇOK PARANIN TOPLANDIĞI İLK 5 ALICI HESAP:")
suspicious_receivers.head(5)

🚨 EN ÇOK PARANIN TOPLANDIĞI İLK 5 ALICI HESAP:


,Receiver_Account_ID,Gelen_Islem_Adedi,Toplam_Gelen_Tutar
25,ACC1025,12,51655.28
16,ACC1016,14,38329.99
9,ACC1009,12,37912.81
34,ACC1034,12,37046.78
0,ACC1000,11,35810.25


In [9]:
# 4. AML Senaryosu: İşlem Türlerine Göre Risk Özeti
risk_by_type = df.groupby('Transaction_Type').agg(
    Islem_Sayisi=('Transaction_ID', 'count'),
    Toplam_Hacim=('Transaction_Amount', 'sum'),
    Ortalama_Hacim=('Transaction_Amount', 'mean')
).reset_index().sort_values(by='Toplam_Hacim', ascending=False)

print("📊 İŞLEM TÜRLERİNE GÖRE RİSK DAĞILIMI:")
risk_by_type

📊 İŞLEM TÜRLERİNE GÖRE RİSK DAĞILIMI:


,Transaction_Type,Islem_Sayisi,Toplam_Hacim,Ortalama_Hacim
1,Transfer,246,515012.82,2093.548049
0,Deposit,166,364057.44,2193.117108
2,Withdrawal,88,206184.46,2343.005227
